# Importing libraries

In [1]:
# Basic libraries
import pandas as pd
from datasets import load_dataset
import time
import pickle


# Classification models
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.naive_bayes import MultinomialNB

# Vectorizers
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# Utilities and metrics
from itertools import product
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from memory_profiler import memory_usage

# Preprocessing
import nltk
import re

# Download nltk resources
nltk.download('wordnet')

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Rafael\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

# Setting seeds

In [2]:
s1 = 2
s2 = 3
s3 = 5

seeds = [s1, s2, s3]

# Importing datasets

In [3]:
ds = load_dataset("thien/political", "classifier")

train = ds['train'].to_pandas()
val = ds['eval'].to_pandas()
test = ds['test'].to_pandas()

train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80000 entries, 0 to 79999
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        80000 non-null  int64 
 1   raw_text  80000 non-null  object
 2   filename  80000 non-null  object
 3   text      80000 non-null  object
 4   split     80000 non-null  object
 5   label     80000 non-null  object
dtypes: int64(1), object(5)
memory usage: 3.7+ MB


# Dataset preprocessing

In [4]:
train = train[['text', 'label']]
val = val[['text', 'label']]
test = test[['text', 'label']]

train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80000 entries, 0 to 79999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    80000 non-null  object
 1   label   80000 non-null  object
dtypes: object(2)
memory usage: 1.2+ MB


In [5]:
stop_words = set(nltk.corpus.stopwords.words('english'))
lemmatizer = nltk.stem.WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()

    text = re.sub(r'[^\w\s]', '', text)
    
    words = text.split()
    words = [word for word in words if word not in stop_words]
    words = [lemmatizer.lemmatize(word) for word in words]

    return ' '.join(words)
train['text'] = train['text'].apply(preprocess_text)

# GridSearch implementation

In [6]:
vectorizers = [
    TfidfVectorizer(),
    CountVectorizer()
]

models = {
    'RandomForest': {
        'model': RandomForestClassifier(),
        'params': {
            'n_estimators': [50, 100, 150],
            'criterion': ['gini', 'entropy', 'log_loss']
        }
    },
    'SVC': {
        'model': SVC(),
        'params': {
            'C': [0.1, 1, 10],
            'kernel': ['linear', 'rbf', 'sigmoid']
        }
    },
    'MultinomialNB': {
        'model': MultinomialNB(),
        'params': {
            'alpha': [0.01, 0.1, 1.0]
        }
    },
    'LogisticRegression': {
        'model': LogisticRegression(max_iter=1000),
        'params': {
            'C': [0.1, 1, 10],
            'penalty': ['l2']
        }
    },
    'KNeighbors': {
        'model': KNeighborsClassifier(),
        'params': {
            'n_neighbors': [3, 5, 7],
            'algorithm': ['ball_tree', 'kd_tree', 'brute']
        }
    },
    'DecisionTree': {
        'model': DecisionTreeClassifier(),
        'params': {
            'criterion': ['gini', 'entropy', 'log_loss'],
            'max_features': ['sqrt', 'log2', None]
        }
    },
    'GradientBoosting': {
        'model': GradientBoostingClassifier(),
        'params': {
            'n_estimators': [100, 150, 200],
            'criterion': ['friedman_mse', 'squared_error'],
        }
    },
    'AdaBoost': {
        'model': AdaBoostClassifier(),
        'params': {
            'n_estimators': [50, 100, 150],
            'learning_rate': [0.01, 0.1, 1.0]
        }
    },
    'SGD': {
        'model': SGDClassifier(),
        'params': {
            'alpha': [0.0001, 0.001, 0.01],
            'penalty': ['l2', 'l1', 'elasticnet']
        }
    }
}

In [7]:
columns = ['seed', 'vectorizer', 'model', 'params', 'accuracy', 'training_time', 'prediction_time', 'peak_memory_train', 'peak_memory_prediction']

classes = sorted(train['label'].unique())
for c in classes:
    columns.extend([
        f'precision_class_{c}',
        f'recall_class_{c}',
        f'f1_class_{c}'
    ])

results = pd.DataFrame(columns=columns)

In [8]:
for seed in seeds:
    print(f"Processing seed: {seed}")
    for vectorizer in vectorizers:
        print(f"Processing vectorizer: {vectorizer.__class__.__name__}")
        for name, info in models.items():
            print(f"Processing model: {name}")

            model = info['model']
            param_grid = info['params']
            param_combinations = product(*param_grid.values())
            
            for combination in param_combinations:
                params = dict(zip(param_grid.keys(), combination))
                model.set_params(**params)
                if 'random_state' in model.get_params():
                    model.set_params(random_state=seed)

                print(f"Training {name} with params {params} and vectorizer {vectorizer.__class__.__name__}")

                pipeline = Pipeline([
                    ('vectorizer', vectorizer),
                    ('model', model)
                ])

                def train_model():
                    pipeline.fit(train['text'], train['label'])

                def predict_model():
                    return pipeline.predict(val['text'])

                # Training Phase
                start_time = time.perf_counter()
                peak_memory_train = memory_usage(train_model, max_usage=True)
                train_time = time.perf_counter() - start_time
                print(f"Training time: {train_time}")
                print(f"Peak memory usage during training: {peak_memory_train} MB")

                # Prediction Phase
                start_time = time.perf_counter()
                peak_memory_pred, y_pred = memory_usage(predict_model, max_usage=True, retval=True)
                prediction_time = time.perf_counter() - start_time
                print(f"Prediction time: {prediction_time}")
                print(f"Peak memory usage during prediction: {peak_memory_pred} MB")
                
                accuracy = accuracy_score(val['label'], y_pred)
                precisions, recalls, f1s, supports = precision_recall_fscore_support(val['label'], y_pred, average=None, labels=classes, zero_division=0)

                result_dict = {
                    'seed': seed,
                    'vectorizer': vectorizer.__class__.__name__,
                    'model': name,
                    'params': params,
                    'accuracy': accuracy,
                    'training_time': train_time,
                    'prediction_time': prediction_time,
                    'peak_memory_train': peak_memory_train,
                    'peak_memory_prediction': peak_memory_pred,
                }

                for i, c in enumerate(classes):
                    result_dict[f'precision_class_{c}'] = precisions[i]
                    result_dict[f'recall_class_{c}'] = recalls[i]
                    result_dict[f'f1_class_{c}'] = f1s[i]

                result = pd.DataFrame([result_dict])
                results = pd.concat([results, result], ignore_index=True)
                print("-"*100)

results.to_csv('results/results_sklearn_binary3.csv', index=False)

Processing seed: 2
Processing vectorizer: TfidfVectorizer
Processing model: RandomForest
Training RandomForest with params {'n_estimators': 50, 'criterion': 'gini'} and vectorizer TfidfVectorizer
Training time: 49.92307339981198
Peak memory usage during training: 608.8984375 MB
Prediction time: 1.2106924999970943
Peak memory usage during prediction: 603.57421875 MB
----------------------------------------------------------------------------------------------------
Training RandomForest with params {'n_estimators': 50, 'criterion': 'entropy'} and vectorizer TfidfVectorizer


C:\Users\Rafael\AppData\Local\Temp\ipykernel_12868\2750059118.py:66: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, result], ignore_index=True)


Training time: 45.51909630000591
Peak memory usage during training: 610.55859375 MB
Prediction time: 1.1890758001245558
Peak memory usage during prediction: 581.609375 MB
----------------------------------------------------------------------------------------------------
Training RandomForest with params {'n_estimators': 50, 'criterion': 'log_loss'} and vectorizer TfidfVectorizer
Training time: 45.98057230003178
Peak memory usage during training: 605.8359375 MB
Prediction time: 1.2111456000711769
Peak memory usage during prediction: 581.3984375 MB
----------------------------------------------------------------------------------------------------
Training RandomForest with params {'n_estimators': 100, 'criterion': 'gini'} and vectorizer TfidfVectorizer
Training time: 99.16859339992516
Peak memory usage during training: 745.375 MB
Prediction time: 0.7300742999650538
Peak memory usage during prediction: 745.67578125 MB
---------------------------------------------------------------------

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\neighbors\_base.py:583: UserWarning: cannot use tree with sparse input: using brute force
  warnings.warn("cannot use tree with sparse input: using brute force")


Training time: 0.9089448000304401
Peak memory usage during training: 844.06640625 MB
Prediction time: 4.315642199944705
Peak memory usage during prediction: 2838.9921875 MB
----------------------------------------------------------------------------------------------------
Training KNeighbors with params {'n_neighbors': 3, 'algorithm': 'kd_tree'} and vectorizer TfidfVectorizer


...

Training time: 13.715483100153506
Peak memory usage during training: 909.90234375 MB
Prediction time: 1.0847824001684785
Peak memory usage during prediction: 871.65625 MB
----------------------------------------------------------------------------------------------------
Training AdaBoost with params {'n_estimators': 150, 'learning_rate': 0.1} and vectorizer CountVectorizer


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


Training time: 13.693845000118017
Peak memory usage during training: 909.578125 MB
Prediction time: 1.0805531996302307
Peak memory usage during prediction: 871.43359375 MB
----------------------------------------------------------------------------------------------------
Training AdaBoost with params {'n_estimators': 150, 'learning_rate': 1.0} and vectorizer CountVectorizer


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


Training time: 13.706735200248659
Peak memory usage during training: 909.96875 MB
Prediction time: 1.0705761001445353
Peak memory usage during prediction: 871.40625 MB
----------------------------------------------------------------------------------------------------
Processing model: SGD
Training SGD with params {'alpha': 0.0001, 'penalty': 'l2'} and vectorizer CountVectorizer
Training time: 0.9397352999076247
Peak memory usage during training: 906.59375 MB
Prediction time: 0.9749640999361873
Peak memory usage during prediction: 902.17578125 MB
----------------------------------------------------------------------------------------------------
Training SGD with params {'alpha': 0.0001, 'penalty': 'l1'} and vectorizer CountVectorizer
Training time: 1.0067627998068929
Peak memory usage during training: 906.53125 MB
Prediction time: 1.4403043002821505
Peak memory usage during prediction: 871.54296875 MB
------------------------------------------------------------------------------------

# Process results

In [9]:
results.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 396 entries, 0 to 395
Data columns (total 15 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   seed                        396 non-null    object 
 1   vectorizer                  396 non-null    object 
 2   model                       396 non-null    object 
 3   params                      396 non-null    object 
 4   accuracy                    396 non-null    float64
 5   training_time               396 non-null    float64
 6   prediction_time             396 non-null    float64
 7   peak_memory_train           396 non-null    float64
 8   peak_memory_prediction      396 non-null    float64
 9   precision_class_democratic  396 non-null    float64
 10  recall_class_democratic     396 non-null    float64
 11  f1_class_democratic         396 non-null    float64
 12  precision_class_republican  396 non-null    float64
 13  recall_class_republican     396 non

In [10]:
results.head()

,seed,vectorizer,model,params,accuracy,training_time,prediction_time,peak_memory_train,peak_memory_prediction,precision_class_democratic,recall_class_democratic,f1_class_democratic,precision_class_republican,recall_class_republican,f1_class_republican
0,2,TfidfVectorizer,RandomForest,"{'n_estimators': 50, 'criterion': 'gini'}",0.94300,49.923073,1.210692,608.898438,603.574219,0.951120,0.9340,0.942482,0.935167,0.9520,0.943508
1,2,TfidfVectorizer,RandomForest,"{'n_estimators': 50, 'criterion': 'entropy'}",0.94275,45.519096,1.189076,610.558594,581.609375,0.953405,0.9310,0.942069,0.932584,0.9545,0.943415
2,2,TfidfVectorizer,RandomForest,"{'n_estimators': 50, 'criterion': 'log_loss'}",0.94275,45.980572,1.211146,605.835938,581.398438,0.953405,0.9310,0.942069,0.932584,0.9545,0.943415
3,2,TfidfVectorizer,RandomForest,"{'n_estimators': 100, 'criterion': 'gini'}",0.94225,99.168593,0.730074,745.375000,745.675781,0.954289,0.9290,0.941475,0.930833,0.9555,0.943005
4,2,TfidfVectorizer,RandomForest,"{'n_estimators': 100, 'criterion': 'entropy'}",0.94650,90.635450,0.722163,751.750000,721.429688,0.961260,0.9305,0.945630,0.932655,0.9625,0.947343


In [11]:
results['params'] = results['params'].astype(str)
results_avg_seed = results.groupby(['model', 'vectorizer', 'params']).mean().reset_index()
results_avg_seed['f1_avg'] = results_avg_seed[[col for col in results_avg_seed.columns if 'f1_class' in col]].mean(axis=1)
results_avg_seed

,model,vectorizer,params,seed,accuracy,training_time,prediction_time,peak_memory_train,peak_memory_prediction,precision_class_democratic,recall_class_democratic,f1_class_democratic,precision_class_republican,recall_class_republican,f1_class_republican,f1_avg
0,AdaBoost,CountVectorizer,"{'n_estimators': 100, 'learning_rate': 0.01}",3.333333,0.61200,9.442402,1.047595,907.980469,871.428385,0.962810,0.2330,0.375201,0.563709,0.9910,0.718637,0.546919
1,AdaBoost,CountVectorizer,"{'n_estimators': 100, 'learning_rate': 0.1}",3.333333,0.72525,9.409267,1.045796,908.072917,871.589844,0.952764,0.4740,0.633055,0.649917,0.9765,0.780420,0.706737
2,AdaBoost,CountVectorizer,"{'n_estimators': 100, 'learning_rate': 1.0}",3.333333,0.82325,9.404816,1.042994,908.135417,871.440104,0.966787,0.6695,0.791137,0.747228,0.9770,0.846804,0.818971
3,AdaBoost,CountVectorizer,"{'n_estimators': 150, 'learning_rate': 0.01}",3.333333,0.63350,13.751020,1.083501,908.225260,871.329427,0.968421,0.2760,0.429572,0.577843,0.9910,0.730018,0.579795
4,AdaBoost,CountVectorizer,"{'n_estimators': 150, 'learning_rate': 0.1}",3.333333,0.75575,13.692523,1.077666,907.938802,871.298177,0.962060,0.5325,0.685549,0.676806,0.9790,0.800327,0.742938
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,SVC,TfidfVectorizer,"{'C': 1, 'kernel': 'rbf'}",3.333333,0.96050,326.041834,5.910294,1137.243490,866.954427,0.970859,0.9495,0.960061,0.950587,0.9715,0.960930,0.960495
128,SVC,TfidfVectorizer,"{'C': 1, 'kernel': 'sigmoid'}",3.333333,0.95750,234.855735,2.452000,1125.575521,860.997396,0.971649,0.9425,0.956853,0.944175,0.9725,0.958128,0.957490
129,SVC,TfidfVectorizer,"{'C': 10, 'kernel': 'linear'}",3.333333,0.95575,163.741871,1.728122,1221.123698,858.875000,0.975483,0.9350,0.954812,0.937590,0.9765,0.956650,0.955731
130,SVC,TfidfVectorizer,"{'C': 10, 'kernel': 'rbf'}",3.333333,0.96025,454.242138,5.939008,1118.397135,863.945312,0.973752,0.9460,0.959675,0.947496,0.9745,0.960808,0.960242


In [12]:
best_result = results_avg_seed.loc[results_avg_seed['f1_avg'].idxmax()]

best_model = models[best_result['model']]['model']
best_params = eval(best_result['params'])
best_model.set_params(**best_params)

best_result_vectorizer = eval(best_result['vectorizer'])()

pipeline = Pipeline([
    ('vectorizer', best_result_vectorizer),
    ('model', best_model)
])

pipeline.fit(train['text'], train['label'])
y_pred = pipeline.predict(test['text'])

accuracy = accuracy_score(test['label'], y_pred)
precisions, recalls, f1s, supports = precision_recall_fscore_support(test['label'], y_pred, average=None, labels=classes, zero_division=0)

print(f"Best model: {best_result['model']}")
print(f"Best model params: {best_result['params']}")
print(f"Best vectorizer: {best_result['vectorizer']}")
print(f"Best accuracy: {accuracy}\n")

for i, c in enumerate(classes):
    print(f"Class {c}")
    print(f"Precision: {precisions[i]}")
    print(f"Recall: {recalls[i]}")
    print(f"F1: {f1s[i]}")
    print(f"Support: {supports[i]}\n")

Best model: MultinomialNB
Best model params: {'alpha': 1.0}
Best vectorizer: CountVectorizer
Best accuracy: 0.9593035714285715

Class democratic
Precision: 0.9678416821273964
Recall: 0.9501785714285714
F1: 0.9589287966984448
Support: 28000

Class republican
Precision: 0.9510715162568834
Recall: 0.9684285714285714
F1: 0.9596715683672206
Support: 28000



In [13]:
with open('models/best_model_sklearn_binary3.pkl', 'wb') as f:
    pickle.dump(pipeline, f)